In [1]:
from google.colab import files
uploaded = files.upload()

Saving departments.csv to departments.csv
Saving employees.csv to employees.csv
Saving employees_nested.json to employees_nested.json
Saving sales.csv to sales.csv


In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("PySpark Lab").getOrCreate()
employees = spark.read.csv("employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("sales.csv", header=True, inferSchema=True)
employees_nested = spark.read.json("employees_nested.json")

employees.printSchema()
sales.printSchema()
departments.printSchema()
employees_nested.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- joining_date: date (nullable = true)

root
 |-- sale_id: integer (nullable = true)
 |-- emp_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)

root
 |-- department: string (nullable = true)
 |-- location: string (nullable = true)

root
 |-- _corrupt_record: string (nullable = true)



In [4]:
employees.show(5)
employees.count()
employees.columns
employees.show(3)

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
+------+-----+----------+------+------------+
only showing top 5 rows
+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
+------+-----+----------+------+------------+
only showing top 3 rows


In [7]:
employees.select("name", "salary").show()
employees.withColumnRenamed("salary", "emp_salary").show()
employees.filter("joining_date > '2023-01-01'").show()
employees.filter("salary > 65000").show()
employees.filter("department = 'IT'").show()
employees.filter("department = 'IT' AND salary > 70000").show()
employees.orderBy("salary").show()
employees.orderBy(employees.salary.desc()).show()
employees.orderBy(employees.salary.desc()).limit(3).show()

+-----+------+
| name|salary|
+-----+------+
|Rahul| 70000|
|Sneha| 60000|
|Arjun| 75000|
|Priya| 80000|
|Karan| 50000|
|Meera| 72000|
| Amit| 58000|
+-----+------+

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     1|Rahul|        IT|     70000|  2023-01-10|
|     2|Sneha|        HR|     60000|  2022-11-15|
|     3|Arjun|        IT|     75000|  2023-03-01|
|     4|Priya|   Finance|     80000|  2022-12-20|
|     5|Karan|        IT|     50000|  2023-02-05|
|     6|Meera|      NULL|     72000|  2023-04-10|
|     7| Amit|        HR|     58000|  2023-01-18|
+------+-----+----------+----------+------------+

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6

In [9]:
from pyspark.sql.functions import sum, avg, max, min
employees.select(sum("salary")).show()
employees.select(avg("salary")).show()
employees.select(max("salary")).show()
employees.select(min("salary")).show()
employees.groupBy("department").count().show()
employees.groupBy("department").agg(avg("salary")).show()
employees.groupBy("department").agg(sum("salary")).show()

+-----------+
|sum(salary)|
+-----------+
|     465000|
+-----------+

+-----------------+
|      avg(salary)|
+-----------------+
|66428.57142857143|
+-----------------+

+-----------+
|max(salary)|
+-----------+
|      80000|
+-----------+

+-----------+
|min(salary)|
+-----------+
|      50000|
+-----------+

+----------+-----+
|department|count|
+----------+-----+
|        HR|    2|
|      NULL|    1|
|   Finance|    1|
|        IT|    3|
+----------+-----+

+----------+-----------+
|department|avg(salary)|
+----------+-----------+
|        HR|    59000.0|
|      NULL|    72000.0|
|   Finance|    80000.0|
|        IT|    65000.0|
+----------+-----------+

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|        HR|     118000|
|      NULL|      72000|
|   Finance|      80000|
|        IT|     195000|
+----------+-----------+



In [8]:
employees.join(departments, "department", "inner").show()
employees.join(departments, "department", "left").show()
employees.join(departments, "department").select("name", "location").show()


+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|        IT|     1|Rahul| 70000|  2023-01-10|Bangalore|
|        HR|     2|Sneha| 60000|  2022-11-15|   Mumbai|
|        IT|     3|Arjun| 75000|  2023-03-01|Bangalore|
|   Finance|     4|Priya| 80000|  2022-12-20|    Delhi|
|        IT|     5|Karan| 50000|  2023-02-05|Bangalore|
|        HR|     7| Amit| 58000|  2023-01-18|   Mumbai|
+----------+------+-----+------+------------+---------+

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|        IT|     1|Rahul| 70000|  2023-01-10|Bangalore|
|        HR|     2|Sneha| 60000|  2022-11-15|   Mumbai|
|        IT|     3|Arjun| 75000|  2023-03-01|Bangalore|
|   Finance|     4|Priya| 80000|  2022-12-20|    Delhi|
|        IT|     5|Karan| 50000|  2023-02-05|Ba

In [12]:
from pyspark.sql.functions import col,lit
employees = employees.withColumn("bonus", col("salary") * 0.10)
employees = employees.withColumn("company", lit("BotCampus"))
employees.show()

+------+-----+----------+------+------------+------+---------+
|emp_id| name|department|salary|joining_date| bonus|  company|
+------+-----+----------+------+------------+------+---------+
|     1|Rahul|        IT| 70000|  2023-01-10|7000.0|BotCampus|
|     2|Sneha|        HR| 60000|  2022-11-15|6000.0|BotCampus|
|     3|Arjun|        IT| 75000|  2023-03-01|7500.0|BotCampus|
|     4|Priya|   Finance| 80000|  2022-12-20|8000.0|BotCampus|
|     5|Karan|        IT| 50000|  2023-02-05|5000.0|BotCampus|
|     6|Meera|      NULL| 72000|  2023-04-10|7200.0|BotCampus|
|     7| Amit|        HR| 58000|  2023-01-18|5800.0|BotCampus|
+------+-----+----------+------+------------+------+---------+



In [13]:
employees.filter(col("department").isNull()).show()
employees.fillna({"department": "Unknown"}).show()
employees.dropna().show()


+------+-----+----------+------+------------+------+---------+
|emp_id| name|department|salary|joining_date| bonus|  company|
+------+-----+----------+------+------------+------+---------+
|     6|Meera|      NULL| 72000|  2023-04-10|7200.0|BotCampus|
+------+-----+----------+------+------------+------+---------+

+------+-----+----------+------+------------+------+---------+
|emp_id| name|department|salary|joining_date| bonus|  company|
+------+-----+----------+------+------------+------+---------+
|     1|Rahul|        IT| 70000|  2023-01-10|7000.0|BotCampus|
|     2|Sneha|        HR| 60000|  2022-11-15|6000.0|BotCampus|
|     3|Arjun|        IT| 75000|  2023-03-01|7500.0|BotCampus|
|     4|Priya|   Finance| 80000|  2022-12-20|8000.0|BotCampus|
|     5|Karan|        IT| 50000|  2023-02-05|5000.0|BotCampus|
|     6|Meera|   Unknown| 72000|  2023-04-10|7200.0|BotCampus|
|     7| Amit|        HR| 58000|  2023-01-18|5800.0|BotCampus|
+------+-----+----------+------+------------+------+--

In [14]:
from pyspark.sql.functions import upper,length
employees.select(upper("name")).show()
employees.select(length("name")).show()
employees.withColumn("salary_in_lakhs", col("salary")/100000).show()


+-----------+
|upper(name)|
+-----------+
|      RAHUL|
|      SNEHA|
|      ARJUN|
|      PRIYA|
|      KARAN|
|      MEERA|
|       AMIT|
+-----------+

+------------+
|length(name)|
+------------+
|           5|
|           5|
|           5|
|           5|
|           5|
|           5|
|           4|
+------------+

+------+-----+----------+------+------------+------+---------+---------------+
|emp_id| name|department|salary|joining_date| bonus|  company|salary_in_lakhs|
+------+-----+----------+------+------------+------+---------+---------------+
|     1|Rahul|        IT| 70000|  2023-01-10|7000.0|BotCampus|            0.7|
|     2|Sneha|        HR| 60000|  2022-11-15|6000.0|BotCampus|            0.6|
|     3|Arjun|        IT| 75000|  2023-03-01|7500.0|BotCampus|           0.75|
|     4|Priya|   Finance| 80000|  2022-12-20|8000.0|BotCampus|            0.8|
|     5|Karan|        IT| 50000|  2023-02-05|5000.0|BotCampus|            0.5|
|     6|Meera|      NULL| 72000|  2023-04-10|72

In [ ]:
employees.withColumn(
    "salary_grade",
    when(col("salary") >= 75000, "High")
    .when((col("salary") >= 60000) & (col("salary") < 75000), "Medium")
    .otherwise("Low")
).show()